# 00/환경 설정 (Setup): 문서 요약

**요약**: AWS/HF 자격증명/리전/역할을 확인하고 의존성을 설치합니다.

**목적**: 이후 모든 노트북이 여기서 만든 config/자격증명을 공유합니다.

**배경**: 노트북마다 설정을 반복하면 값이 서로 어긋나는 config drift가 발생하므로, 모든 설정을 common/config.py 한 곳에서 관리합니다.

> 실제 실행에는 AWS 자격증명과 비용이 필요합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

## 1. 의존성 설치 (uv 권장/pip 폴백)
재현 가능한 환경을 위해 의존성 관리는 `uv`를 우선 사용하고, 설치되어 있지 않은 경우 `pip`으로 폴백합니다. `uv`가 없다면 다음 명령으로 먼저 설치하세요: `curl -LsSf https://astral.sh/uv/install.sh | sh`
- 의존성을 최신 버전으로 올리려면 `uv lock --upgrade` 후 `uv sync`를 실행합니다. 특정 패키지만 갱신할 때는 `uv lock --upgrade-package transformers`처럼 지정합니다.

In [ ]:
import shutil, subprocess, sys, os
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if shutil.which('uv'):
    # uv가 현재 노트북 커널 인터프리터(sys.executable)에 pyproject.toml 의존성 설치. 최신은 uv lock --upgrade.
    subprocess.run(['uv', 'pip', 'install', '--python', sys.executable, '-r', 'pyproject.toml'], cwd=REPO, check=True)
else:
    print('uv not found -> pip fallback. (recommended: curl -LsSf https://astral.sh/uv/install.sh | sh)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    os.path.join(REPO, 'requirements.txt')], check=True)

## 2. 경로 설정: common/ import
노트북은 트랙 하위 디렉토리에서 실행되므로, 리포 루트와 트랙 로컬 경로를 `sys.path`에 추가해야 `common/` 공용 모듈과 트랙별 `track_data` 모듈을 import할 수 있습니다.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

## 3. 환경변수 (플레이스홀더: 시크릿 하드코딩 금지)
리전/모델 ID 등 실행에 필요한 값을 환경변수로 설정합니다. HF 토큰이나 AWS 자격증명 같은 시크릿을 노트북에 직접 하드코딩하면 유출 위험이 있으므로, 아래에서는 플레이스홀더만 두고 실제 값은 env로 주입합니다. `setdefault`를 사용해 이미 설정된 값이 있으면 덮어쓰지 않습니다.

In [ ]:
# setdefault보다 먼저 저장소 루트의 .env를 읽습니다.
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(REPO, '.env'))
except ImportError:
    print('python-dotenv가 없어 .env를 읽지 못했습니다. `uv sync`로 설치하세요.')

# 셸과 .env에 값이 없을 때만 기본 리전을 사용합니다.
os.environ.setdefault('AWS_REGION', 'us-west-2')
# boto3 기본 세션도 같은 리전을 사용하도록 맞춥니다.
os.environ['AWS_DEFAULT_REGION'] = os.environ['AWS_REGION']
print('AWS_REGION =', os.environ['AWS_REGION'])

# 모델 크기: 'E2B' | 'E4B'(기본, 단일 GPU) | '12B' | '26B-A4B' | '31B'
os.environ.setdefault('MODEL_SIZE', 'E4B')
os.environ.setdefault('DRY_RUN', '1')   # 파이프라인 검증용. 실제 실행 시 '0'

# (선택) 필요할 때만 주석 해제
# os.environ['MODEL_ID'] = 'google/gemma-4-12B-it'      # 프리셋 대신 직접 지정
# os.environ['MODEL_IS_GATED'] = '1'                    # gemma-3/2 등 gated 모델
# os.environ['HF_HOME'] = os.path.expanduser('~/hf-cache')   # hf login 을 이 경로로 했다면
# os.environ['BEDROCK_CLAUDE_MODEL_ID'] = 'us.anthropic.claude-...'

> **위 (선택) 항목 참고**
> - **HF 토큰**: gemma-4는 ungated라 필요 없습니다. gated 모델을 쓸 때만 `MODEL_IS_GATED=1` + 토큰. 토큰은 env `HF_TOKEN` 또는 `hf auth login` 저장분을 config가 자동으로 찾습니다. 단 `hf auth login`을 커스텀 경로로 했다면 `HF_HOME`도 같이 맞춰야 합니다.
> - **Bedrock 모델 ID**: inference-profile 접두사(`us.`/`global.` 등)가 붙은 정확한 ID여야 합니다: 콘솔 모델 상세 페이지에서 확인하세요.

## 4. 로깅 설정 (1회) & config 로드
로깅 핸들러는 애플리케이션 진입점에서 한 번만 구성하는 것이 원칙입니다. 라이브러리 코드(common/*)가 핸들러를 직접 설정하면 로그가 중복 출력되거나 사용자 설정을 덮어쓰기 때문에, common/* 모듈은 핸들러를 건드리지 않고 노트북에서 `setup_logging()`으로 1회 구성합니다.

In [ ]:
from common.logging_utils import setup_logging, get_logger
setup_logging()          # LOG_LEVEL env 존중(기본 INFO), 멱등
log = get_logger('nb')   # gemma_e2e.nb
log.info('Logging configured')

In [ ]:
import importlib
from common import config; importlib.reload(config)
TRACK = config.TRACKS['summarization']
print('TRACK          :', TRACK.name)
print('seed_dataset   :', TRACK.seed_dataset)
print('MODEL_ID       :', config.DEFAULT_MODEL_ID)
print('AWS_REGION     :', config.AWS_REGION)
print('DRY_RUN        :', config.is_dry_run())
print('HF_TOKEN set   :', bool(config.get_hf_token()))

In [ ]:
import boto3
try:
    ident = boto3.client('sts', region_name=config.AWS_REGION).get_caller_identity()
    print('AWS account:', ident['Account'])
except Exception as e:
    print('WARNING: AWS credential check failed - run aws configure or set a role:', e)

# 리전 일치 확인: boto3 '기본 세션'(리전 미지정 클라이언트가 쓰는 것)도 같은 리전인지 본다.
#    다르면 SDK 내부가 region_name 없이 만드는 클라이언트가 옛 리전에 붙을 수 있다.
_default = boto3.Session().region_name
print('boto3 default  :', _default)
if _default != config.AWS_REGION:
    print(f"주의:  boto3 기본 리전({_default}) != AWS_REGION({config.AWS_REGION}).")
    print('    위 셀의 AWS_DEFAULT_REGION 설정이 실행됐는지 확인하세요(커널 재시작 후 순서대로 실행).')
    print(f'    ~/.aws/config 의 [default] region 이 {_default} 로 돼 있어도 이 env가 우선합니다.')
else:
    print('리전 일치: SDK 내부 클라이언트도 같은 리전을 씁니다.')

## 5. SageMaker 세션 & 역할
SageMaker 학습 잡과 endpoint 배포에는 세션 객체와 실행 IAM role, 그리고 아티팩트를 저장할 S3 bucket이 필요합니다. role은 `config.resolve_sagemaker_role()`이 **env `SAGEMAKER_ROLE_ARN` → `get_execution_role()`(Studio/NB) → IAM 자동 탐지(AmazonSageMaker-ExecutionRole-*)** 순으로 해석하므로, IAM user로 로컬 실행해도 계정의 실행 role을 자동으로 찾습니다. role ARN을 코드나 `.env`에 하드코딩하지 마세요: 특정 계정에 종속되고 이식성이 떨어집니다(명시가 필요하면 셸에서 `export SAGEMAKER_ROLE_ARN=...`). 여기서 확인한 role/bucket은 `%store`로 저장해 이후 노트북에서 재사용합니다.

In [ ]:
# sagemaker SDK v3(3.x): Session/get_execution_role 는 sagemaker.core.helper.session_helper 로 이동.
from sagemaker.core.helper.session_helper import Session
sess = Session(boto3.Session(region_name=config.AWS_REGION))
role = config.resolve_sagemaker_role(sess)   # env → get_execution_role → IAM 자동 탐지
bucket = config.S3_BUCKET or sess.default_bucket()
print('role  :', role); print('bucket:', bucket)
%store role
%store bucket

환경 설정이 끝났습니다. 다음은 **01_data_and_synthetic.ipynb**로 이어집니다.